# Vessel Routes Finder — Anomaly Detection & Illegal Fishing Activity

**Goal:** Build an end-to-end data-science pipeline that ingests vessel tracking
(AIS) data, engineers behavioural features, and flags *anomalous* vessel
behaviour that may indicate **illegal, unreported, and unregulated (IUU)
fishing** or other suspicious activity (AIS "dark" gaps, loitering,
at-sea rendezvous / transshipment, fishing inside Marine Protected Areas).

**Data source:** [Global Fishing Watch (GFW) API](https://globalfishingwatch.org/our-apis/)
— a free (non-commercial) source of AIS-derived fishing-effort, vessel,
and event data covering the global fishing fleet.

### What this notebook covers
1. Setup & dependencies
2. Connecting to the GFW API (with a token)
3. **Live data pull for Syria** (vessels by flag + events/effort in the EEZ)
4. A **synthetic AIS generator** so the whole notebook runs *without* a token
5. Exploratory analysis & route mapping
6. Feature engineering (speed, turning, AIS gaps, distance-to-port, zone tests)
7. Unsupervised anomaly detection (Isolation Forest, LOF, DBSCAN)
8. Rule-based detectors for known IUU patterns
9. Visualising & interpreting the flagged tracks
10. Evaluation and next steps

> The pipeline is built so you can swap the synthetic data for real GFW data
> by setting one flag once you have an API token.


## 1. Setup & dependencies

Run this once. If a package is missing, the `pip install` line will fetch it.

In [ ]:


%pip install -q pandas numpy scikit-learn matplotlib seaborn folium requests geopy python-dotenv

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN

pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Environment ready.")


## 2. Connecting to the Global Fishing Watch API

GFW offers several REST APIs (all share one bearer token):

| API | What it gives you |
|-----|-------------------|
| **4Wings** | Gridded AIS apparent-fishing-effort rasters & time series |
| **Vessels** | Search vessels by name / MMSI / IMO, get identity & history |
| **Events** | Discrete events: `fishing`, `encounter` (rendezvous), `loitering`, `port_visit`, `gap` (AIS off) |
| **Insights** | Risk indicators per vessel |

### Get a token (free, ~1 day approval)
1. Register at <https://globalfishingwatch.org/our-apis/>
2. Create an **API access token** in your account.
3. Provide it in any of these ways (checked in this order) — never commit it:
   * a **`.env`** file in the project root: `GFW_API_TOKEN=your_token` (git-ignored, auto-loaded)
   * an exported **environment variable**: `export GFW_API_TOKEN=your_token`
   * (last resort) paste directly into the config cell below.

The `Events` API is the most useful for IUU work — `encounter`, `loitering`
and `gap` events are *exactly* the behaviours investigators care about.


In [ ]:
import os, requests, json

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

GFW_API_TOKEN = os.environ.get("GFW_API_TOKEN", "")
USE_REAL_GFW_DATA = bool(GFW_API_TOKEN)
GFW_BASE = "https://gateway.api.globalfishingwatch.org/v3"

HEADERS = {"Authorization": f"Bearer {GFW_API_TOKEN}"} if GFW_API_TOKEN else {}

print("Mode:", "REAL GFW DATA" if USE_REAL_GFW_DATA else "SYNTHETIC DATA (no token found)")


In [ ]:
def gfw_get(path, params=None):
    """Thin wrapper around GFW v3 GET endpoints with basic error handling."""
    if not GFW_API_TOKEN:
        raise RuntimeError("No GFW_API_TOKEN set — using synthetic data instead.")
    r = requests.get(f"{GFW_BASE}{path}", headers=HEADERS, params=params, timeout=90)
    r.raise_for_status()
    return r.json()

def gfw_post(path, body, params=None):
    """POST wrapper for GFW v3 endpoints that take a JSON body (events, 4wings report)."""
    if not GFW_API_TOKEN:
        raise RuntimeError("No GFW_API_TOKEN set — using synthetic data instead.")
    headers = {**HEADERS, "Content-Type": "application/json"}
    r = requests.post(f"{GFW_BASE}{path}", headers=headers, params=params, json=body, timeout=120)
    r.raise_for_status()
    return r.json()

def gfw_search_vessels(query, limit=5):
    """Search the vessel registry by free text (name / MMSI / IMO / callsign)."""
    params = {"query": query, "datasets[0]": "public-global-vessel-identity:latest",
              "limit": limit}
    return gfw_get("/vessels/search", params)

if USE_REAL_GFW_DATA:
    try:
        demo = gfw_get("/vessels/search",
                       {"where": "flag = \'SYR\'",
                        "datasets[0]": "public-global-vessel-identity:latest",
                        "limit": 3})
        print("Connection OK. Sample response keys:", list(demo.keys()))
        print(json.dumps(demo, indent=2)[:800])
    except Exception as e:
        print("API call failed:", e)
        print("If this says 'Host not in allowlist', your environment's network policy")
        print("is blocking globalfishingwatch.org — run locally or widen the allowlist.")
        USE_REAL_GFW_DATA = False
else:
    print("Skipping live call — no token. The notebook will use synthetic data below.")


## 3. Live data pull — Syria

This section uses your token to pull **real data for Syria** from GFW:

1. **Vessels flagged to Syria** (`flag = 'SYR'`) from the vessel-identity registry.
2. **Events inside the Syrian EEZ** — `fishing`, `encounter`, `loitering`, `gap`
   — queried by a GeoJSON polygon over Syrian waters (Levantine Sea).
3. **Apparent fishing effort** (4Wings report) aggregated over the same area.

> **Network note:** if you see `Host not in allowlist` / a 403, the machine you're
> running on is blocking `globalfishingwatch.org`. Run the notebook on a machine
> with normal internet (or widen your environment's network allowlist). The cells
> are written to *degrade gracefully* — they print the error and the notebook
> continues to the synthetic demo below.

> **EEZ box:** we use a bounding-box polygon over Syrian waters. For production,
> swap in the official Syria EEZ polygon (Marine Regions / `public-eez-areas`).


In [ ]:

SYRIA_FLAG = "SYR"

SYR_BBOX = {"lat_min": 34.55, "lat_max": 35.95, "lon_min": 33.50, "lon_max": 36.05}
SYR_POLYGON = {
    "type": "Polygon",
    "coordinates": [[
        [SYR_BBOX["lon_min"], SYR_BBOX["lat_min"]],
        [SYR_BBOX["lon_max"], SYR_BBOX["lat_min"]],
        [SYR_BBOX["lon_max"], SYR_BBOX["lat_max"]],
        [SYR_BBOX["lon_min"], SYR_BBOX["lat_max"]],
        [SYR_BBOX["lon_min"], SYR_BBOX["lat_min"]],
    ]],
}
DATE_START, DATE_END = "2023-01-01", "2023-12-31"

def safe(fn, *args, **kwargs):
    """Run an API call, returning None and printing a hint on failure."""
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        msg = str(e)
        print(f"  x {getattr(fn, '__name__', fn)} failed: {msg[:160]}")
        if "allowlist" in msg or "403" in msg:
            print("    -> network policy is blocking GFW; run where internet is open.")
        return None


In [ ]:

def gfw_vessels_by_flag(flag=SYRIA_FLAG, limit=100):
    params = {
        "where": f"flag = \'{flag}\'",
        "datasets[0]": "public-global-vessel-identity:latest",
        "limit": limit,
    }
    return gfw_get("/vessels/search", params)

def vessels_to_frame(resp):
    """Flatten a GFW vessel-search response into a tidy DataFrame (defensive)."""
    rows = []
    for entry in (resp or {}).get("entries", []):

        sri = (entry.get("selfReportedInfo") or [{}])
        reg = (entry.get("registryInfo") or [{}])
        src = sri[0] if sri else {}
        rg  = reg[0] if reg else {}
        rows.append({
            "vessel_id":  entry.get("dataset") and (src.get("id") or rg.get("id")) or entry.get("id"),
            "shipname":   src.get("shipname") or rg.get("shipname"),
            "flag":       src.get("flag") or rg.get("flag"),
            "ssvid_mmsi": src.get("ssvid") or rg.get("ssvid"),
            "imo":        src.get("imo") or rg.get("imo"),
            "callsign":   src.get("callsign") or rg.get("callsign"),
            "geartype":   (src.get("geartypes") or rg.get("geartypes")),
            "from":       src.get("transmissionDateFrom"),
            "to":         src.get("transmissionDateTo"),
        })
    return pd.DataFrame(rows)

syria_vessels = pd.DataFrame()
if USE_REAL_GFW_DATA:
    resp = safe(gfw_vessels_by_flag, SYRIA_FLAG, 100)
    syria_vessels = vessels_to_frame(resp)
    print(f"Syria-flagged vessels retrieved: {len(syria_vessels)}")

else:
    print("No token / synthetic mode — skipping Syria vessel pull.")

syria_vessels.head(20)


In [ ]:


EVENT_DATASETS = {
    "fishing":    "public-global-fishing-events:latest",
    "encounter":  "public-global-encounters-events:latest",
    "loitering":  "public-global-loitering-events:latest",
    "gap":        "public-global-gaps-events:latest",
}

def gfw_events_in_region(dataset, geometry=SYR_POLYGON,
                         start=DATE_START, end=DATE_END, limit=1000):
    body = {
        "datasets": [dataset],
        "startDate": start,
        "endDate": end,
        "geometry": geometry,
    }
    return gfw_post("/events", body, params={"limit": limit, "offset": 0})

def events_to_frame(resp, etype):
    rows = []
    for ev in (resp or {}).get("entries", []):
        pos = ev.get("position") or {}
        ves = ev.get("vessel") or {}
        rows.append({
            "type": etype,
            "event_id": ev.get("id"),
            "start": ev.get("start"), "end": ev.get("end"),
            "lat": pos.get("lat"), "lon": pos.get("lon"),
            "vessel_id": ves.get("id"), "vessel_name": ves.get("name"),
            "vessel_flag": ves.get("flag"), "ssvid": ves.get("ssvid"),
        })
    return pd.DataFrame(rows)

syria_events = pd.DataFrame()
if USE_REAL_GFW_DATA:
    frames = []
    for etype, ds in EVENT_DATASETS.items():
        resp = safe(gfw_events_in_region, ds)
        df = events_to_frame(resp, etype)
        print(f"  {etype:10s}: {len(df)} events")
        frames.append(df)
    syria_events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print(f"Total Syrian-EEZ events: {len(syria_events)}")
else:
    print("No token / synthetic mode — skipping Syria events pull.")

syria_events.head(20)


In [ ]:

def gfw_fishing_effort(geometry=SYR_POLYGON, start=DATE_START, end=DATE_END):
    params = {
        "spatial-resolution": "LOW",
        "temporal-resolution": "MONTHLY",
        "datasets[0]": "public-global-fishing-effort:latest",
        "date-range": f"{start},{end}",
        "format": "JSON",
        "group-by": "FLAG",
    }
    body = {"geojson": geometry}
    return gfw_post("/4wings/report", body, params=params)

syria_effort = None
if USE_REAL_GFW_DATA:
    syria_effort = safe(gfw_fishing_effort)
    if syria_effort:
        print("4Wings report keys:", list(syria_effort.keys()))
        print(json.dumps(syria_effort, indent=2)[:1000])
else:
    print("No token / synthetic mode — skipping fishing-effort pull.")


### Feed real Syria events into the detector (optional)

If `syria_events` came back populated, the event records (`encounter`,
`loitering`, `gap`) are *already* the suspicious behaviours we care about — you
can map them straight onto the same anomaly framework used below, or plot them
on the Syria map. The synthetic section that follows demonstrates the full
modelling pipeline regardless of network access.


In [ ]:
if USE_REAL_GFW_DATA and len(syria_events):
    try:
        import folium
        m = folium.Map(location=[35.30, 35.50], zoom_start=7, tiles="CartoDB positron")
        folium.Rectangle([(SYR_BBOX["lat_min"], SYR_BBOX["lon_min"]),
                          (SYR_BBOX["lat_max"], SYR_BBOX["lon_max"])],
                         color="green", fill=False, tooltip="Syria EEZ (approx)").add_to(m)
        colours = {"fishing": "blue", "encounter": "red", "loitering": "orange", "gap": "black"}
        for _, r in syria_events.dropna(subset=["lat", "lon"]).iterrows():
            folium.CircleMarker([r["lat"], r["lon"]], radius=4,
                                color=colours.get(r["type"], "gray"), fill=True,
                                tooltip=f"{r['type']} — {r.get('vessel_name')}").add_to(m)
        display(m)
    except Exception as e:
        print("Map skipped:", e)
else:
    print("No live Syria events to map (need token + open network).")


## 4. Synthetic AIS generator (so the notebook always runs)

Real AIS data are sequences of **positional pings** per vessel:
`(mmsi, timestamp, lat, lon, speed, course)`. Below we simulate a small fleet.
Most vessels behave "normally" (steam out, fish in legal grounds, steam home),
but we deliberately inject a few **anomalous** vessels exhibiting classic IUU
signatures so we can later check whether our models catch them:

* **`dark_gap`** — vessel switches off AIS for a long stretch (a "dark" period).
* **`mpa_fishing`** — vessel loiters/fishes inside a protected zone.
* **`rendezvous`** — two vessels meet at sea at low speed (possible transshipment).

Each ping is labelled with a ground-truth `is_anomalous` flag for evaluation —
but the detection models below never see that label (fully unsupervised).


In [ ]:
from datetime import datetime, timedelta

MPA = {"lat_min": 2.0, "lat_max": 4.0, "lon_min": -8.0, "lon_max": -6.0}
PORT = (0.0, 0.0)

def _walk(lat, lon, course_deg, dist_km):
    """Move a point dist_km along a bearing (small-distance flat-earth approx)."""
    dlat = (dist_km / 111.0) * np.cos(np.radians(course_deg))
    dlon = (dist_km / (111.0 * np.cos(np.radians(lat)))) * np.sin(np.radians(course_deg))
    return lat + dlat, lon + dlon

def simulate_vessel(mmsi, behaviour="normal", n=240, start=None):
    """Generate one vessel's AIS track (a ping every 30 min)."""
    start = start or datetime(2022, 6, 1)
    lat, lon = PORT
    course = np.random.uniform(20, 70)
    rows = []
    for i in range(n):
        ts = start + timedelta(minutes=30 * i)
        phase = "transit" if i < n * 0.25 or i > n * 0.75 else "fishing"
        if phase == "transit":
            speed = np.random.normal(11, 1.2)
            course += np.random.normal(0, 4)
        else:
            speed = np.random.normal(3.5, 1.0)
            course += np.random.normal(0, 35)
        anomalous = False

        if behaviour == "mpa_fishing" and phase == "fishing":
            lat = np.clip(lat, MPA["lat_min"], MPA["lat_max"])
            lon = np.clip(lon, MPA["lon_min"], MPA["lon_max"])
            anomalous = True
        if behaviour == "rendezvous" and phase == "fishing" and n * 0.45 < i < n * 0.55:
            speed = np.random.normal(0.4, 0.2)
            anomalous = True
        if behaviour == "dark_gap" and n * 0.4 < i < n * 0.6:
            continue

        speed = max(speed, 0)
        rows.append((mmsi, ts, round(lat, 5), round(lon, 5),
                     round(speed, 2), round(course % 360, 1), behaviour, anomalous))
        lat, lon = _walk(lat, lon, course, speed * 0.5 * 1.852)
    cols = ["mmsi", "timestamp", "lat", "lon", "speed", "course", "behaviour", "is_anomalous"]
    return pd.DataFrame(rows, columns=cols)

fleet = []
for m in range(1, 13):
    fleet.append(simulate_vessel(100000000 + m, "normal"))
fleet.append(simulate_vessel(200000001, "dark_gap"))
fleet.append(simulate_vessel(200000002, "mpa_fishing"))
fleet.append(simulate_vessel(200000003, "rendezvous"))

ais = pd.concat(fleet, ignore_index=True).sort_values(["mmsi", "timestamp"])
print(f"{ais.mmsi.nunique()} vessels, {len(ais):,} AIS pings")
ais.head()


## 5. Exploratory analysis & route mapping

First, look at the raw tracks and the speed distribution that separates *steaming* from *fishing*.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for mmsi, g in ais.groupby("mmsi"):
    color = "crimson" if g["behaviour"].iloc[0] != "normal" else "steelblue"
    lw = 2.0 if color == "crimson" else 0.8
    axes[0].plot(g["lon"], g["lat"], color=color, lw=lw, alpha=0.7)

import matplotlib.patches as patches
axes[0].add_patch(patches.Rectangle(
    (MPA["lon_min"], MPA["lat_min"]),
    MPA["lon_max"] - MPA["lon_min"], MPA["lat_max"] - MPA["lat_min"],
    fill=True, alpha=0.15, color="green", label="Marine Protected Area"))
axes[0].scatter(*PORT[::-1], c="black", marker="*", s=200, label="Port", zorder=5)
axes[0].set(title="Vessel tracks (red = injected anomalies)", xlabel="Longitude", ylabel="Latitude")
axes[0].legend(loc="upper left")

sns.histplot(ais["speed"], bins=40, ax=axes[1], color="slateblue")
axes[1].axvline(4, color="red", ls="--", label="~fishing speed threshold")
axes[1].set(title="Speed distribution (knots)", xlabel="Speed")
axes[1].legend()
plt.tight_layout(); plt.show()


### Interactive map with Folium (optional)

Folium renders a Leaflet map you can pan/zoom. Great for inspecting a single
suspicious track.


In [ ]:
try:
    import folium
    center = [ais.lat.mean(), ais.lon.mean()]
    fmap = folium.Map(location=center, zoom_start=5, tiles="CartoDB positron")

    folium.Rectangle([(MPA["lat_min"], MPA["lon_min"]), (MPA["lat_max"], MPA["lon_max"])],
                     color="green", fill=True, fill_opacity=0.1, tooltip="MPA").add_to(fmap)

    for mmsi, g in ais.groupby("mmsi"):
        anomalous = g["behaviour"].iloc[0] != "normal"
        folium.PolyLine(list(zip(g["lat"], g["lon"])),
                        color="red" if anomalous else "blue",
                        weight=3 if anomalous else 1.5, opacity=0.7,
                        tooltip=f"MMSI {mmsi} ({g['behaviour'].iloc[0]})").add_to(fmap)
    display(fmap)
except Exception as e:
    print("Folium not available or rendering skipped:", e)


## 6. Feature engineering

Anomaly models are only as good as their features. From the raw ping stream we
derive, **per ping** (and aggregated per vessel), signals that capture *how* a
vessel is moving:

* `speed`, `accel` — instantaneous speed and its change
* `turn` — absolute heading change (erratic turning ⇒ fishing/searching)
* `gap_minutes` — time since previous ping (large ⇒ possible AIS "dark" period)
* `jump_km` — distance since previous ping (implausible jumps ⇒ spoofing/gap)
* `dist_to_port_km` — far-from-port + slow can mean transshipment/loitering
* `in_mpa` — is the ping inside a protected area?


In [ ]:
from geopy.distance import great_circle

def add_features(df):
    df = df.sort_values(["mmsi", "timestamp"]).copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    df["gap_minutes"] = df.groupby("mmsi")["timestamp"].diff().dt.total_seconds() / 60
    prev = df.groupby("mmsi")[["lat", "lon"]].shift(1)
    df["jump_km"] = [
        great_circle((a, b), (c, d)).km if pd.notnull(c) else 0.0
        for a, b, c, d in zip(df["lat"], df["lon"], prev["lat"], prev["lon"])
    ]

    df["accel"] = df.groupby("mmsi")["speed"].diff().abs().fillna(0)
    course_diff = df.groupby("mmsi")["course"].diff().abs().fillna(0)
    df["turn"] = np.minimum(course_diff, 360 - course_diff)

    df["dist_to_port_km"] = [great_circle((la, lo), PORT).km for la, lo in zip(df["lat"], df["lon"])]
    df["in_mpa"] = ((df["lat"].between(MPA["lat_min"], MPA["lat_max"])) &
                    (df["lon"].between(MPA["lon_min"], MPA["lon_max"]))).astype(int)

    df["gap_minutes"] = df["gap_minutes"].fillna(30)
    return df

ais_f = add_features(ais)
FEATURES = ["speed", "accel", "turn", "gap_minutes", "jump_km", "dist_to_port_km", "in_mpa"]
ais_f[FEATURES].describe().round(2)


## 7. Unsupervised anomaly detection

We don't usually have labelled "this vessel was illegal" data, so we lean on
**unsupervised** detectors that learn the shape of *normal* behaviour and flag
points that don't fit:

* **Isolation Forest** — isolates outliers with random splits; fast, robust, scales well.
* **Local Outlier Factor (LOF)** — density-based; flags points in sparse neighbourhoods.
* **DBSCAN** — clusters dense behaviour; leftover noise points (`-1`) are anomalies.

We scale features first (these models are distance/extent sensitive).


In [ ]:
X = StandardScaler().fit_transform(ais_f[FEATURES])

iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=42)
ais_f["iso_flag"] = (iso.fit_predict(X) == -1).astype(int)
ais_f["iso_score"] = -iso.score_samples(X)

lof = LocalOutlierFactor(n_neighbors=35, contamination=0.05)
ais_f["lof_flag"] = (lof.fit_predict(X) == -1).astype(int)

db = DBSCAN(eps=1.5, min_samples=10).fit(X)
ais_f["dbscan_flag"] = (db.labels_ == -1).astype(int)

ais_f["anomaly_votes"] = ais_f[["iso_flag", "lof_flag", "dbscan_flag"]].sum(axis=1)
ais_f["anomaly"] = (ais_f["anomaly_votes"] >= 2).astype(int)

print(ais_f[["iso_flag", "lof_flag", "dbscan_flag", "anomaly"]].sum())


## 8. Rule-based detectors for known IUU patterns

Machine learning is complemented by explicit, explainable rules investigators trust:

In [ ]:
def rule_flags(df):
    df = df.copy()

    df["flag_dark_gap"] = (df["gap_minutes"] > 90).astype(int)

    df["flag_jump"]     = (df["jump_km"] > 40).astype(int)

    df["flag_mpa"]      = ((df["in_mpa"] == 1) & (df["speed"] < 4)).astype(int)

    df["flag_loiter"]   = ((df["speed"] < 1.0) & (df["dist_to_port_km"] > 50)).astype(int)
    return df

ais_f = rule_flags(ais_f)
rule_cols = ["flag_dark_gap", "flag_jump", "flag_mpa", "flag_loiter"]
ais_f[rule_cols].sum()


In [ ]:

ais_f["suspicion"] = ais_f["anomaly"] + ais_f[rule_cols].sum(axis=1)

vessel_risk = (ais_f.groupby("mmsi")
               .agg(behaviour=("behaviour", "first"),
                    pings=("mmsi", "size"),
                    ml_anomaly_pings=("anomaly", "sum"),
                    dark_gaps=("flag_dark_gap", "sum"),
                    mpa_hits=("flag_mpa", "sum"),
                    loiter_hits=("flag_loiter", "sum"),
                    suspicion=("suspicion", "sum"))
               .sort_values("suspicion", ascending=False))
vessel_risk


## 9. Visualising the flagged points

Overlay the points our pipeline flagged on the fleet map — do they land on the bad actors?

In [ ]:
plt.figure(figsize=(9, 7))
for mmsi, g in ais_f.groupby("mmsi"):
    plt.plot(g["lon"], g["lat"], color="lightgray", lw=0.7, zorder=1)

flagged = ais_f[ais_f["suspicion"] > 0]
plt.scatter(flagged["lon"], flagged["lat"], c=flagged["suspicion"],
            cmap="autumn_r", s=25, zorder=3, label="flagged pings")
plt.colorbar(label="suspicion score")
plt.gca().add_patch(patches.Rectangle(
    (MPA["lon_min"], MPA["lat_min"]),
    MPA["lon_max"] - MPA["lon_min"], MPA["lat_max"] - MPA["lat_min"],
    fill=True, alpha=0.15, color="green"))
plt.scatter(*PORT[::-1], c="black", marker="*", s=200, zorder=5)
plt.title("Pipeline-flagged AIS pings"); plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.legend(); plt.tight_layout(); plt.show()


## 10. Evaluation

Because the synthetic generator labelled every ping with the ground-truth
`is_anomalous` flag, we can measure how well the *unsupervised* pipeline
recovered the injected anomalies. (With real GFW data you'd instead validate
against known IUU cases, watchlists, or expert review.)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_true = ais_f["is_anomalous"].astype(int)
y_pred = (ais_f["suspicion"] > 0).astype(int)

print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"]))
print("Confusion matrix [ [TN FP] [FN TP] ]:")
print(confusion_matrix(y_true, y_pred))
try:
    print(f"\nIsolationForest score ROC-AUC: {roc_auc_score(y_true, ais_f['iso_score']):.3f}")
except Exception as e:
    print("ROC-AUC skipped:", e)


## 11. Where to take this next

**Make it real**
* Plug in your **GFW token** and replace the synthetic fleet with real
  `Events` data (`encounter`, `loitering`, `gap`) and `4Wings` fishing effort.
* Add static context layers: **EEZ boundaries**, **real MPA shapefiles**
  (Marine Regions / Protected Planet), RFMO areas — use `geopandas` +
  point-in-polygon tests instead of the toy bounding box.

**Stronger models**
* **Sequence models** that see a track as a time series:
  LSTM/GRU **autoencoders** or **Transformers** that reconstruct normal tracks
  and flag high reconstruction error.
* **Trajectory clustering** (e.g. on resampled fixed-length tracks) to find
  fleets behaving alike.
* Calibrate `contamination` / thresholds against a labelled validation set if
  you can obtain one (e.g. known IUU vessel lists).

**Productionise**
* Schedule daily GFW pulls, store in a database, and serve a dashboard
  (Streamlit / Folium) that ranks vessels by risk.
* Track model performance over time and let analysts label false positives to
  improve thresholds.

**Responsible use**
* AIS gaps and loitering are *indicators*, not proof — present results as
  **leads for human review**, document assumptions, and beware false positives
  (legitimate reasons exist for slow speed, AIS outages, etc.).

---
*Data: Global Fishing Watch. Built as an educational reference pipeline.*
